# Detailed variant statistics
Here we get detailed variant statistics for variant groups of interest

In [1]:
# Make sure our package is installed (run once)
#!pip install -e /home/dataproc/repos/igvf-cvfg-biobank

Notes on running in AoU's DataProc: as of 2026-09-25, trying to use pandas with the `fastparquet` engine throws an error. Manually forcing an update of relevant packages resolves it:

In [2]:
# Run once in dataproc
#!pip install --upgrade numpy numba fastparquet pandas statsmodels

In the standard environment, `fastparquet` is not installed, but we use it to append to a parquet file:

In [3]:
# Run once in a standard env
#!pip install fastparquet

In [4]:
import os
import pandas as pd
from pathlib import Path
from functools import cache
import hail as hl
from cvfgaou import notation, hailtools
from tqdm.notebook import tqdm

In [5]:
# In standard env
#wgs_mt_path = "/home/jupyter/workspace/All of Us Controlled Tier v8/vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/splitMT/hail.mt"

In [6]:
# In Dataproc
wgs_mt_path = 'gs://vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/splitMT/hail.mt'

In [7]:
# Constants (mostly paths to things)

variant_table_path = 'variant_tables/integrated_variant_effect_biobank_input_data.20260917.tsv.gz'
gene_cohorts_path = 'gene_cohorts'

google_project = os.getenv('GOOGLE_PROJECT')

functional_variant_counts_path = 'variant_groups/functional-variant-counts_2026-10-10.parquet'
excalibr_variant_counts_path = 'variant_groups/functional-variant-counts_2026-10-10.parquet'
vep_variant_counts_path = 'variant_groups/functional-variant-counts_2026-10-10.parquet'
tp_variant_counts_path = 'variant_groups/functional-variant-counts_2026-10-10.parquet'

In [8]:
# Load dataframe
variants_df = pd.read_table(variant_table_path, dtype=str)

In [9]:
variants_df

Now establish the sets of variants to test.

First we inspect the columns that will define variant sets:

In [10]:
variants_df[['condensed_consequence']].value_counts(dropna=False)

In [11]:
variants_df[['auth_reported_func_class_category']].value_counts(dropna=False)

In [12]:
variants_df['ExCALIBR_points'].value_counts(dropna=False)

In [13]:
# Limit to genes with case/control cohorts established
genes_with_cohorts = {
    gene for gene in variants_df['Gene'].unique()
    if Path(gene_cohorts_path, f'{gene} cohort.parquet').exists()
}

In [14]:
genes_with_cohorts

Function for retrieving cohort hail table efficiently

In [28]:
@cache
def get_cohort_ht(gene):
    cohort_df = pd.read_parquet(Path(gene_cohorts_path, f'{gene} cohort.parquet'))
    cohort_df['s'] = cohort_df.index.to_series().astype(str)
    return hl.Table.from_pandas(
        cohort_df,
        key='s'
    ).cache()

Function for counting variants in AoU cohort broken up by case/control 

In [32]:
def get_variant_counts(
    variant_filtered_mt,
    cohort_ht
):
    cohort_mt = variant_filtered_mt.semi_join_cols(cohort_ht)
    cohort_mt = cohort_mt.annotate_cols(case=cohort_ht.index(cohort_mt.s).case)
    cohort_mt = cohort_mt.annotate_rows(
        in_cases=hl.agg.any(cohort_mt.GT.is_non_ref() & (cohort_mt.case == 1)),
        in_controls=hl.agg.any(cohort_mt.GT.is_non_ref() & (cohort_mt.case == 0))
    )
    return cohort_mt.aggregate_rows(hl.dict([
        ('Variants in WGS MT', hl.agg.count()),
        ('Variants in cohort', hl.agg.count_where(cohort_mt.in_cases | cohort_mt.in_controls)),
        ('Case only variant count', hl.agg.count_where(cohort_mt.in_cases & ~cohort_mt.in_controls)),
        ('Control only variant count', hl.agg.count_where(~cohort_mt.in_cases & cohort_mt.in_controls)),
        ('Overlap variant count', hl.agg.count_where(cohort_mt.in_cases & cohort_mt.in_controls))
    ]))
    

## Functional classes

For each dataset, we examine the following variant sets are by functional category:

- Normal function (all)
- Abnormal function:
  - All
  - Missense
  - Truncating
  - Inframe indel
  - Splicing
  - Non-truncating
  - Non-missense

In [17]:
variant_groups = {
    (dataset, gene, 'Functional class', consequence, classification):
        df[classification_selection]
    for (dataset, gene), df in variants_df.groupby(['Dataset', 'Gene'])
    for classification, consequence, classification_selection in (
        ('Functionally Normal', 'All', df['auth_reported_func_class_category'] == 'normal'),
        ('Functionally Abnormal', 'All', df['auth_reported_func_class_category'] == 'abnormal'),
        ('Functionally Abnormal', 'Missense',
         (df['auth_reported_func_class_category'] == 'abnormal') &
         (df['condensed_consequence'] == 'missense_variant')
        ),
        ('Functionally Abnormal', 'Truncating',
         (df['auth_reported_func_class_category'] == 'abnormal') &
         (df['condensed_consequence'] == 'stop_gained')
        ),
        ('Functionally Abnormal', 'Inframe indel',
         (df['auth_reported_func_class_category'] == 'abnormal') &
         (df['condensed_consequence'] == 'inframe_indel')
        ),
        ('Functionally Abnormal', 'Splicing',
         (df['auth_reported_func_class_category'] == 'abnormal') &
         df['condensed_consequence'].isin(('splicing_variant', 'splice_site_variant'))
        ),
        ('Functionally Abnormal', 'Non-truncating',
         (df['auth_reported_func_class_category'] == 'abnormal') &
         ~(df['condensed_consequence'] == 'stop_gained')
        ),
        ('Functionally Abnormal', 'Non-missense',
         (df['auth_reported_func_class_category'] == 'abnormal') &
         ~(df['condensed_consequence'] == 'missense_variant')
        )
    )
    if gene in genes_with_cohorts
}

Next, get the counts for each variant group

In [18]:
# First, make sure the position column is an integer and
# make sure that the chromosomes are strings with the "chr" prefix

variant_groups = {
    group_label: df.assign(
        Chromosome = 'chr' + df['Chrom'],
        pos = pd.to_numeric(df['hg38_start'], downcast='integer')
    )
    for group_label, df in variant_groups.items()
}

In [19]:
# Check for missing NT coordinates
for label, df in variant_groups.items():
    if df[['Chromosome', 'pos', 'ref_allele', 'alt_allele']].isna().any(axis=None):
        display(label)
        display(df)

In [20]:
# Save variant sets for each group
variant_group_counts = {
    group_label: df[['Chromosome', 'pos', 'ref_allele', 'alt_allele']].drop_duplicates().shape[0]
    for group_label, df in variant_groups.items()
}

In [21]:
# Get direct variant group counts
variant_counts_df = pd.Series(variant_group_counts).to_frame(name="Variants in group")

In [22]:
# Drop empty groups
variant_groups = {
    group_label: df
    for group_label, df in variant_groups.items()
    if not df.empty
}

Run the counter functions

In [23]:
hl.init(
    #default_reference='GRCh38', deprecated
    gcs_requester_pays_configuration=google_project
)

In [24]:
# WGS table

wgs_mt = hl.read_matrix_table(wgs_mt_path)

In [33]:
functional_variant_counts_dict = {
    group_label: get_variant_counts(
        hailtools.get_filtered_mt(
            df[['Chromosome', 'pos', 'ref_allele', 'alt_allele']], wgs_mt,
            contig_col='Chromosome',
            pos_col='pos',
            ref_col='ref_allele',
            alt_col='alt_allele',
            reference_genome='GRCh38'
        ),
        get_cohort_ht(group_label[1]) # Gene is the 2nd field
    )
    for group_label, df in tqdm(variant_groups.items())
}

In [34]:
functional_variant_counts_df = pd.DataFrame.from_dict(
    functional_variant_counts_dict,
    orient='index'
)

In [35]:
pd.concat(
    [variant_counts_df, functional_variant_counts_df],
    axis='columns'
).to_parquet(functional_variant_counts_path)

## ExCALIBR

In [37]:
variant_groups = {
    (dataset, gene, 'ExCALIBR', consequence, classification):
        df[classification_selection & consequence_selection]
    for (dataset, gene), df in variants_df.groupby(['Dataset', 'Gene'])
    for classification, classification_selection in [
        (f'{notation.GEQ_CHAR} {points:+d}', pd.to_numeric(df['ExCALIBR_points']) >= points)
        for points in range(1,9)
    ] + [
        (f'{notation.LEQ_CHAR} {-points:+d}', pd.to_numeric(df['ExCALIBR_points']) <= -points)
        for points in range(1,9)
    ]
    for consequence, consequence_selection in (
        ('All', True),
        ('Missense', df['condensed_consequence'] == 'missense_variant'),
        ('Stop Gained', df['condensed_consequence'] == 'stop_gained')
    )
    if gene in genes_with_cohorts
}

Next, get the counts for each variant group

In [38]:
# First, make sure the position column is an integer and
# make sure that the chromosomes are strings with the "chr" prefix

variant_groups = {
    group_label: df.assign(
        Chromosome = 'chr' + df['Chrom'],
        pos = pd.to_numeric(df['hg38_start'], downcast='integer')
    )
    for group_label, df in variant_groups.items()
}

In [39]:
# Check for missing NT coordinates
for label, df in variant_groups.items():
    if df[['Chromosome', 'pos', 'ref_allele', 'alt_allele']].isna().any(axis=None):
        display(label)
        display(df)

In [40]:
# Save variant sets for each group
variant_group_counts = {
    group_label: df[['Chromosome', 'pos', 'ref_allele', 'alt_allele']].drop_duplicates().shape[0]
    for group_label, df in variant_groups.items()
}

In [41]:
# Get direct variant group counts
variant_counts_df = pd.Series(variant_group_counts).to_frame(name="Variants in group")

In [42]:
# Drop empty groups
variant_groups = {
    group_label: df
    for group_label, df in variant_groups.items()
    if not df.empty
}

Run the counter functions

In [44]:
# WGS table

wgs_mt = hl.read_matrix_table(wgs_mt_path)

In [ ]:
excalibr_variant_counts_dict = {
    group_label: get_variant_counts(
        hailtools.get_filtered_mt(
            df[['Chromosome', 'pos', 'ref_allele', 'alt_allele']], wgs_mt,
            contig_col='Chromosome',
            pos_col='pos',
            ref_col='ref_allele',
            alt_col='alt_allele',
            reference_genome='GRCh38'
        ),
        get_cohort_ht(group_label[1]) # Gene is the 2nd field
    )
    for group_label, df in tqdm(variant_groups.items())
}

In [ ]:
excalibr_variant_counts_df = pd.DataFrame.from_dict(
    excalibr_variant_counts_dict,
    orient='index'
)

In [ ]:
pd.concat(
    [variant_counts_df, excalibr_variant_counts_df],
    axis='columns'
).to_parquet(excalibr_variant_counts_path)

## Combined points

In [ ]:
variant_groups = {
    ('Combined Points', gene, 'ExCALIBR + REVEL single-gene', consequence, classification):
        df[classification_selection & consequence_selection]
    for gene, df in variants_df.groupby('Gene')
    for classification, classification_selection in [
        (f'{notation.GEQ_CHAR} {points:+d}', pd.to_numeric(df['combined_points']) >= points)
        for points in range(1,13)
    ] + [
        (f'{notation.LEQ_CHAR} {-points:+d}', pd.to_numeric(df['combined_points']) <= -points)
        for points in range(1,13)
    ]
    for consequence, consequence_selection in (
        ('All', True),
        ('Missense', df['condensed_consequence'] == 'missense_variant'),
        ('Stop Gained', df['condensed_consequence'] == 'stop_gained')
    )
    if gene in genes_with_cohorts
}